In [ ]:
# Import packages

import polars as pl
from rapidfuzz import fuzz, process

In [18]:
# Import data
df = (pl
      .read_csv(r"C:\Users\knguy139\Documents\Projects\Data\Output\KN_MCR_SS_JOIN_MMID_202509221642.csv")
      .select(["member_id", "work_item_id", "site_clm_aud_nbr"])
)

In [20]:
df.count()

member_id,work_item_id,site_clm_aud_nbr
u32,u32,u32
348689,348689,348689


In [23]:
# Do extra cleaning on work_item_id and site_clm_aud_nbr
# Remove extra whitespace, special characters, ...

df_cleaned = (
    df
    .with_columns([
        pl.col("work_item_id")
            .cast(pl.Utf8)
            .str.strip_chars()
            .str.to_uppercase()
            .str.replace_all(r'[^A-Z0-9]', '')
            .alias('work_item_id_cleaned'), 
        pl.col("site_clm_aud_nbr")
            .cast(pl.Utf8)
            .str.strip_chars()
            .str.to_uppercase()
            .str.replace_all(r'[^A-Z0-9]', '')
            .alias('site_clm_aud_nbr_cleaned')
        ]
    )
)

df_cleaned

member_id,work_item_id,site_clm_aud_nbr,work_item_id_cleaned,site_clm_aud_nbr_cleaned
i64,str,str,str,str
967250197,"""72162847-""","""VRG0067074375""","""72162847""","""VRG0067074375"""
117360508,"""SF20250523611255912""","""23E785923200""","""SF20250523611255912""","""23E785923200"""
967046934,"""79616067-""","""KEN0081457382""","""79616067""","""KEN0081457382"""
967046934,"""79616067-""","""KEN0078031129""","""79616067""","""KEN0078031129"""
967046934,"""79616067-""","""KEN0079061634""","""79616067""","""KEN0079061634"""
…,…,…,…,…
121870645,"""SF20250221552159466""","""23P325867201""","""SF20250221552159466""","""23P325867201"""
809306306,"""53760844-0""","""KEN0076269821""","""537608440""","""KEN0076269821"""
126081667,"""SF20250527612885270""","""24U618412100""","""SF20250527612885270""","""24U618412100"""


In [43]:
# Apply a matching function for each member_id row
# Then assign a grade, and sort by higher to lower grade

df_scored = (
    df_cleaned
    .with_columns([
        pl.struct(['work_item_id_cleaned', 'site_clm_aud_nbr_cleaned'])
        .map_elements(lambda x: int(fuzz.partial_ratio(str(x['work_item_id_cleaned']),str(x['site_clm_aud_nbr_cleaned']))), return_dtype = pl.Int64)
        .alias('match_score')
    ])
    .with_columns([
        pl
        .when(pl.col('match_score') >= 95).then(pl.lit(6))
        .when(pl.col('match_score') >= 90).then(pl.lit(5))
        .when(pl.col('match_score') >= 85).then(pl.lit(4))
        .when(pl.col('match_score') >= 80).then(pl.lit(3))
        .when(pl.col('match_score') >= 75).then(pl.lit(2))
        .when(pl.col('match_score') >= 70).then(pl.lit(1))
        .otherwise(pl.lit(0))
        .alias('match_grade')
    ])
    .sort(['match_grade', 'member_id'], descending = [True, False])
)
df_scored

member_id,work_item_id,site_clm_aud_nbr,work_item_id_cleaned,site_clm_aud_nbr_cleaned,match_score,match_grade
i64,str,str,str,str,i64,i32
110966887,"""25H401937200""","""25H401937200""","""25H401937200""","""25H401937200""",100,6
111827473,"""25H415391200""","""25H415391200""","""25H415391200""","""25H415391200""",100,6
114057918,"""24T785721301""","""24T785721301""","""24T785721301""","""24T785721301""",100,6
114057918,"""24T785721301""","""24T785721301""","""24T785721301""","""24T785721301""",100,6
114063285,"""24T207833100""","""24T207833101""","""24T207833100""","""24T207833101""",95,6
…,…,…,…,…,…,…
997974624,"""52080829-0""","""NYC0096692244""","""520808290""","""NYC0096692244""",40,0
997996643,"""66286639-""","""STL0056587501""","""66286639""","""STL0056587501""",26,0
997996643,"""66286639-""","""STL0056587501""","""66286639""","""STL0056587501""",26,0


In [45]:
# Export to csv
df_scored.write_csv(r"C:\Users\knguy139\Documents\Projects\Data\Output\df_scored.csv")